In [5]:
import requests
from bs4 import BeautifulSoup
import json
import re
from unidecode import unidecode
import os
import unicodedata
from typing import Iterable, Tuple, Any, Dict, Set, List
import shutil
import time

In [6]:
# Read the Google API key from the file
with open('google_api', 'r') as file:
    GOOGLE_API_KEY = file.read().strip()
with open('omdb_api', 'r') as file:
    OMDB_API_KEY = file.read().strip()

In [7]:
## LIFO ###

In [ ]:
def get_sitemap_links(url):
    try:
        # 1. Fetch the sitemap content
        # Adding a User-Agent header helps avoid being blocked by some servers
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # 2. Parse the XML content
        # We use 'xml' (or 'lxml-xml') to ensure it handles XML tags correctly
        soup = BeautifulSoup(response.content, 'xml')
        # 3. Find all <loc> tags (these contain the URLs)
        # Using .text to get the content inside the tags and .strip() to clean it
        links = [loc.text.strip() for loc in soup.find_all('loc')]

        return links

    except Exception as e:
        print(f"An error occurred: {e}")
        return []

# Execute
target_url = "https://www.lifo.gr/sitemap.xml"
all_pages = get_sitemap_links(target_url)

# Print results
print(f"Total links found: {len(all_pages)}")
for link in all_pages:
    print(link)



https://www.lifo.gr/sitemap.xml?page=1
2026-03-16T01:02:02+02:00


https://www.lifo.gr/sitemap.xml?page=2
2026-03-16T01:02:02+02:00


https://www.lifo.gr/sitemap.xml?page=3
2026-03-16T01:17:01+02:00


https://www.lifo.gr/sitemap.xml?page=4
2026-03-16T01:17:01+02:00


https://www.lifo.gr/sitemap.xml?page=5
2026-03-16T01:17:01+02:00


https://www.lifo.gr/sitemap.xml?page=6
2026-03-16T01:32:01+02:00


https://www.lifo.gr/sitemap.xml?page=7
2026-03-16T01:32:01+02:00


https://www.lifo.gr/sitemap.xml?page=8
2026-03-16T01:32:01+02:00


https://www.lifo.gr/sitemap.xml?page=9
2026-03-16T01:47:01+02:00


https://www.lifo.gr/sitemap.xml?page=10
2026-03-16T01:47:01+02:00


https://www.lifo.gr/sitemap.xml?page=11
2026-03-16T01:47:01+02:00


https://www.lifo.gr/sitemap.xml?page=12
2026-03-16T02:02:01+02:00


https://www.lifo.gr/sitemap.xml?page=13
2026-03-16T02:02:01+02:00


https://www.lifo.gr/sitemap.xml?page=14
2026-03-16T02:17:01+02:00


https://www.lifo.gr/sitemap.xml?page=15
2026-03-16T02:1

In [9]:
def get_cinema_links(all_pages):
    base_sitemap_url = "https://www.lifo.gr/sitemap.xml?page="
    target_pattern = "/guide/cinema/movies/"
    all_movie_links = []

    print(f"Starting crawl...")

    for url in all_pages:
        
        print(f"Scanning: {url}")
        
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            
            # Use 'xml' parser for sitemaps
            soup = BeautifulSoup(response.content, 'xml')
            
            # Find all <loc> tags which contain the URLs
            loc_tags = soup.find_all('loc')
            
            page_links = [loc.text for loc in loc_tags if target_pattern in loc.text]
            all_movie_links.extend(page_links)
            
            print(f" Found {len(page_links)} movie links on page {url}.")
            
            # Be polite to the server
            time.sleep(0.1) 
            
        except Exception as e:
            print(f" Error on page {page}: {e}")
            break

    return all_movie_links

# Run the script
movies = get_cinema_links(all_pages)

print("\n--- Extraction Complete ---")
for link in sorted(set(movies)):
    print(link)

Starting crawl...
Scanning: https://www.lifo.gr/sitemap.xml?page=1
 Found 0 movie links on page https://www.lifo.gr/sitemap.xml?page=1.
Scanning: https://www.lifo.gr/sitemap.xml?page=2
 Found 0 movie links on page https://www.lifo.gr/sitemap.xml?page=2.
Scanning: https://www.lifo.gr/sitemap.xml?page=3
 Found 0 movie links on page https://www.lifo.gr/sitemap.xml?page=3.
Scanning: https://www.lifo.gr/sitemap.xml?page=4
 Found 0 movie links on page https://www.lifo.gr/sitemap.xml?page=4.
Scanning: https://www.lifo.gr/sitemap.xml?page=5
 Found 0 movie links on page https://www.lifo.gr/sitemap.xml?page=5.
Scanning: https://www.lifo.gr/sitemap.xml?page=6
 Found 0 movie links on page https://www.lifo.gr/sitemap.xml?page=6.
Scanning: https://www.lifo.gr/sitemap.xml?page=7
 Found 0 movie links on page https://www.lifo.gr/sitemap.xml?page=7.
Scanning: https://www.lifo.gr/sitemap.xml?page=8
 Found 0 movie links on page https://www.lifo.gr/sitemap.xml?page=8.
Scanning: https://www.lifo.gr/sitemap.

In [10]:
lifo_movie_links = sorted(set(movies))

In [11]:
clean_lifo_links = []
for url in lifo_movie_links:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
            
    # Use 'xml' parser for sitemaps
    pattern = re.compile(r'Η ΤΑΙΝΙΑ ΔΕΝ ΠΡΟΒΑΛΛΕΤΑΙ AYTH ΤΗ ΣΤΙΓΜΗ ΣΕ ΚΑΠΟΙΑ ΑΙΘΟΥΣΑ')
    matches = pattern.findall(response.text)
    if matches:
        print('not used',url)
    else:
        print('ok',url)
        clean_lifo_links.append(url)

not used https://www.lifo.gr/guide/cinema/movies/100-lykos
not used https://www.lifo.gr/guide/cinema/movies/1341-kare-erota-kai-polemoy
not used https://www.lifo.gr/guide/cinema/movies/16-fores-anoixi
not used https://www.lifo.gr/guide/cinema/movies/18
not used https://www.lifo.gr/guide/cinema/movies/1917
not used https://www.lifo.gr/guide/cinema/movies/1976
not used https://www.lifo.gr/guide/cinema/movies/200-lykos
not used https://www.lifo.gr/guide/cinema/movies/200-metra
not used https://www.lifo.gr/guide/cinema/movies/20000-eidi-melisson
not used https://www.lifo.gr/guide/cinema/movies/2046
not used https://www.lifo.gr/guide/cinema/movies/2073
not used https://www.lifo.gr/guide/cinema/movies/22-ioylioy
not used https://www.lifo.gr/guide/cinema/movies/28-xronia-meta
not used https://www.lifo.gr/guide/cinema/movies/28-xronia-meta-o-naos-ton-oston
not used https://www.lifo.gr/guide/cinema/movies/47
not used https://www.lifo.gr/guide/cinema/movies/48-ores-stin-taiban
not used https://w

In [12]:
clean_lifo_links

['https://www.lifo.gr/guide/cinema/movies/agios-paisios',
 'https://www.lifo.gr/guide/cinema/movies/amartoloi',
 'https://www.lifo.gr/guide/cinema/movies/amnet',
 'https://www.lifo.gr/guide/cinema/movies/anemodarmena-upsi',
 'https://www.lifo.gr/guide/cinema/movies/anoixi-kalokairi-fthinoporo-heimonas-kai-anoixi',
 'https://www.lifo.gr/guide/cinema/movies/apostoli-haire-maria',
 'https://www.lifo.gr/guide/cinema/movies/aristero-moy-heri',
 'https://www.lifo.gr/guide/cinema/movies/boygonia',
 'https://www.lifo.gr/guide/cinema/movies/drive-my-car',
 'https://www.lifo.gr/guide/cinema/movies/dyo-eisaggeleis',
 'https://www.lifo.gr/guide/cinema/movies/ego-kapetanios',
 'https://www.lifo.gr/guide/cinema/movies/ena-aplo-atyhima',
 'https://www.lifo.gr/guide/cinema/movies/epic-o-elvis-presley-se-mia-monadiki-synaylia',
 'https://www.lifo.gr/guide/cinema/movies/father-mother-sister-brother',
 'https://www.lifo.gr/guide/cinema/movies/filoi-gia-panta',
 'https://www.lifo.gr/guide/cinema/movies/ga

In [13]:
results = []

for url in clean_lifo_links:
    print(url)
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html')
    
    title = ""
    title_tag = soup.find("h1", class_="eventTitle")
    if title_tag:
        title = title_tag.get_text(strip=True)
    
    parent = soup.find('div', class_='lifoRating fs-9-v-lg fs-7-v')  
    rating_number = 0
    if parent:
        # 2. Search ONLY inside that parent for the ratings div
        # 1. Find the div that contains the "rating-" class
        rating_div = parent.find('div', class_='ratings')

        if rating_div:
            # 2. rating_div['class'] returns a list: ['ratings', 'rating-3', 'mb-lg-3', 'mb-2']
            # We look for the item that starts with 'rating-'
            rating_class = next((c for c in rating_div['class'] if c.startswith('rating-')), None)
            rating_number = rating_class.split('-')[-1]
    else:
        print('no rating')
        
    # store result
    results.append({
        "url": url,
        "title": title,
        "rating": rating_number
    })

# save to json
with open("lifo_ratings.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("Saved to lifo_ratings.json")

https://www.lifo.gr/guide/cinema/movies/agios-paisios
no rating
https://www.lifo.gr/guide/cinema/movies/amartoloi
https://www.lifo.gr/guide/cinema/movies/amnet
https://www.lifo.gr/guide/cinema/movies/anemodarmena-upsi
https://www.lifo.gr/guide/cinema/movies/anoixi-kalokairi-fthinoporo-heimonas-kai-anoixi
https://www.lifo.gr/guide/cinema/movies/apostoli-haire-maria
no rating
https://www.lifo.gr/guide/cinema/movies/aristero-moy-heri
https://www.lifo.gr/guide/cinema/movies/boygonia
https://www.lifo.gr/guide/cinema/movies/drive-my-car
https://www.lifo.gr/guide/cinema/movies/dyo-eisaggeleis
https://www.lifo.gr/guide/cinema/movies/ego-kapetanios
https://www.lifo.gr/guide/cinema/movies/ena-aplo-atyhima
https://www.lifo.gr/guide/cinema/movies/epic-o-elvis-presley-se-mia-monadiki-synaylia
https://www.lifo.gr/guide/cinema/movies/father-mother-sister-brother
https://www.lifo.gr/guide/cinema/movies/filoi-gia-panta
https://www.lifo.gr/guide/cinema/movies/galazio-monopati
no rating
https://www.lifo.

In [14]:
### FLIX ####

In [15]:
import requests
from bs4 import BeautifulSoup
import re

def get_flix_review_links():
    search_url = "https://flix.gr/search-movies-in-cinemas/"
    domain = "https://flix.gr"
    
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36',
        'Referer': domain
    })

    try:
        session.get(domain)
        response = session.get(search_url)
        response.raise_for_status()
        
        html_content = response.text
        soup = BeautifulSoup(html_content, 'html.parser')
        
        found_links = set()

        # --- PART 1: Standard href Extraction ---
        for a_tag in soup.find_all('a', href=re.compile(r'-review')):
            href = a_tag['href'].strip()
            # Normalize URL
            if href.startswith('http'):
                full_url = href
            elif href.startswith('/'):
                full_url = f"{domain}{href}"
            else:
                full_url = f"{domain}/cinema/{href}"
            found_links.add(full_url)

        # --- PART 2: Regex search for "url": "..." strings ---
        # This looks for the specific "url": "name-review" pattern in the text
        json_style_pattern = re.compile(r'"url":\s*"([^"]+)"')
        matches = json_style_pattern.findall(html_content)

        for match in matches:
            # We only care if it's a review link
            if "-review" in match:
                # Add .html if it's missing from the string
                clean_match = match if match.endswith('.html') else f"{match}.html"
                
                # Normalize URL
                if clean_match.startswith('http'):
                    full_url = clean_match
                elif clean_match.startswith('/'):
                    full_url = f"{domain}{clean_match}"
                else:
                    full_url = f"{domain}/cinema/{clean_match}"
                
                found_links.add(full_url)
        
        return sorted(list(found_links))

    except Exception as e:
        print(f"An error occurred: {e}")
        return []

# Run
review_list = get_flix_review_links()

print(f"--- Found {len(review_list)} unique review links ---")
for link in review_list:
    print(link)

--- Found 73 unique review links ---
https://flix.gr/cinema/agios-paisios-review.html
https://flix.gr/cinema/arco-review.html
https://flix.gr/cinema/beachcomber-review.html
https://flix.gr/cinema/bernard-mission-mars-review.html
https://flix.gr/cinema/blow-up-review.html
https://flix.gr/cinema/broken-vein-review.html
https://flix.gr/cinema/bugonia-review.html
https://flix.gr/cinema/calle-malaga-review.html
https://flix.gr/cinema/charlie-the-wonderdog-review.html
https://flix.gr/cinema/checkered-ninja-3-review.html
https://flix.gr/cinema/cries-and-whispers-review.html
https://flix.gr/cinema/crime-101-review.html
https://flix.gr/cinema/drive-my-car-review.html
https://flix.gr/cinema/drommer-review.html
https://flix.gr/cinema/epic-elvis-presley-in-concert-review.html
https://flix.gr/cinema/father-mother-sister-brother-review.html
https://flix.gr/cinema/filoi-gia-panta-review.html
https://flix.gr/cinema/goat-review.html
https://flix.gr/cinema/good-luck-have-fun-dont-die-review.html
https:/

In [16]:
flix_movie_links = review_list

In [17]:
import requests
from bs4 import BeautifulSoup
import re

def get_flix_rating(url):
    print(url)
    
    rating = None
    title = None
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, "html.parser")
        
        # --- Get title ---
        title_tag = soup.find("h1")
        movie_title = title_tag.get_text(strip=True) if title_tag else None


        tag = soup.find("span", itemprop="aggregateRating")

        if tag and tag.has_attr("title"):
            title = tag["title"]          # e.g. "8 στα 10"

            match = re.search(r"(\d+)\s*στα\s*10", title)
            if match:
                rating = int(match.group(1))

        return rating,movie_title

    except Exception as e:
        print(f"Error with {url}: {e}")
        return None


results = []

for url in flix_movie_links:
    rating, movie_title = get_flix_rating(url)

    results.append({
        "url": url,
        "title": movie_title,
        "rating": rating
    })

# save json
with open("flix_ratings.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

https://flix.gr/cinema/agios-paisios-review.html
https://flix.gr/cinema/arco-review.html
https://flix.gr/cinema/beachcomber-review.html
https://flix.gr/cinema/bernard-mission-mars-review.html
https://flix.gr/cinema/blow-up-review.html
https://flix.gr/cinema/broken-vein-review.html
https://flix.gr/cinema/bugonia-review.html
https://flix.gr/cinema/calle-malaga-review.html
https://flix.gr/cinema/charlie-the-wonderdog-review.html
https://flix.gr/cinema/checkered-ninja-3-review.html
https://flix.gr/cinema/cries-and-whispers-review.html
https://flix.gr/cinema/crime-101-review.html
https://flix.gr/cinema/drive-my-car-review.html
https://flix.gr/cinema/drommer-review.html
https://flix.gr/cinema/epic-elvis-presley-in-concert-review.html
https://flix.gr/cinema/father-mother-sister-brother-review.html
https://flix.gr/cinema/filoi-gia-panta-review.html
https://flix.gr/cinema/goat-review.html
https://flix.gr/cinema/good-luck-have-fun-dont-die-review.html
https://flix.gr/cinema/hamnet-review.html
ht

In [18]:
import json
import unicodedata
import pandas as pd


# ---------- normalize text ----------
def normalize(text):
    if not text:
        return ""
        
    text = text.lower().strip()

    text = ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )

    return text


# ---------- load files ----------
with open("movies.json", encoding="utf-8") as f:
    ath_data = json.load(f)

with open("flix_ratings.json", encoding="utf-8") as f:
    flix_data = json.load(f)


# ---------- flatten athinorama ----------
ath_movies = []
for group in ath_data:
    for movie in group:
        ath_movies.append(movie)


# ---------- build lookup dictionary ----------
ath_lookup = {}

for m in ath_movies:

    greek_key = normalize(m["greek_title"])
    original_key = normalize(m["original_title"])

    ath_lookup[greek_key] = m
    ath_lookup[original_key] = m


# ---------- match movies ----------
matches = []

for m in flix_data:

    key = normalize(m["title"])

    if key in ath_lookup:

        ath_movie = ath_lookup[key]

        matches.append({
            "title": m["title"],
            "year": ath_movie["year"],
            "flix_rating": m["rating"],
            "ath_rating": ath_movie["rating_stars"]
        })


df = pd.DataFrame(matches)


# ---------- statistics ----------
print("Total Athinorama movies:", len(ath_movies))
print("Total Flix movies:", len(flix_data))
print("Matched movies:", len(df))


print("\nFlix rating stats")
print(df["flix_rating"].describe())


print("\nAthinorama rating stats")
print(df["ath_rating"].describe())


# ---------- correlation between critics ----------
print("\nCorrelation between critics")
print(df[["flix_rating","ath_rating"]].corr())

Total Athinorama movies: 84
Total Flix movies: 73
Matched movies: 55

Flix rating stats
count    55.000000
mean      6.054545
std       2.422259
min       0.000000
25%       5.000000
50%       6.000000
75%       8.000000
max      10.000000
Name: flix_rating, dtype: float64

Athinorama rating stats
count    28.000000
mean      3.357143
std       0.780042
min       2.000000
25%       3.000000
50%       3.000000
75%       4.000000
max       5.000000
Name: ath_rating, dtype: float64

Correlation between critics
             flix_rating  ath_rating
flix_rating     1.000000    0.660596
ath_rating      0.660596    1.000000


In [19]:
import json
import unicodedata
import pandas as pd


# ---------- normalize text ----------
def normalize(text):
    if not text:
        return ""

    text = text.lower().strip()

    text = ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )

    return text


# ---------- load files ----------
with open("movies.json", encoding="utf-8") as f:
    ath_data = json.load(f)

with open("flix_ratings.json", encoding="utf-8") as f:
    flix_data = json.load(f)


# ---------- flatten athinorama ----------
ath_movies = []
for group in ath_data:
    for movie in group:
        ath_movies.append(movie)


# ---------- build lookup ----------
ath_lookup = {}

for m in ath_movies:
    greek_key = normalize(m["greek_title"])
    original_key = normalize(m["original_title"])

    ath_lookup[greek_key] = m
    ath_lookup[original_key] = m


# ---------- match ----------
matches = []
matched_ath_keys = set()
unmatched_flix = []

for m in flix_data:

    key = normalize(m["title"])

    if key in ath_lookup:

        ath_movie = ath_lookup[key]

        matches.append({
            "title": m["title"],
            "year": ath_movie["year"],
            "flix_rating": m["rating"],
            "ath_rating": ath_movie["rating_stars"]
        })

        matched_ath_keys.add(normalize(ath_movie["greek_title"]))
        matched_ath_keys.add(normalize(ath_movie["original_title"]))

    else:
        unmatched_flix.append(m["title"])


# ---------- find athinorama movies not matched ----------
unmatched_ath = []

for m in ath_movies:

    greek_key = normalize(m["greek_title"])
    original_key = normalize(m["original_title"])

    if greek_key not in matched_ath_keys and original_key not in matched_ath_keys:
        unmatched_ath.append(m["greek_title"])


# ---------- print results ----------
print("\nUNMATCHED FLIX MOVIES")
for t in unmatched_flix:
    print("-", t)


print("\nUNMATCHED ATHINORAMA MOVIES")
for t in unmatched_ath:
    print("-", t)


UNMATCHED FLIX MOVIES
- Αρκο
- Μυστικός Πράκτορας Μπέρνι:  Αποστολή στον Άρη
- Blow-Up
- Τσάρλι ο Σούπερ-Σκύλος
- Καρό Νίντζα 3
- Ονειρα (Σεξ, Αγάπη)
- EPiC: Ο Elvis Presley σε μια Μοναδική Συναυλία
- Good Luck, Have Fun, Don't Die
- Μαουτχάουζεν
- Οικογένεια Προς Ενοικίαση
- Scarlet
- Βοήθεια!
- Η Νύφη
- Η Αγγελία
- Το Τελευταίο Σημείωμα
- Μπομπ Σφουγγαράκης: Η Αναζήτηση του Τετραγωνοπαντελόνη!
- Κλείδωσες; Οι Αγνωστοι 3
- Γυναίκες Μαχήτριες Γ' Μέρος (1960-1974)

UNMATCHED ATHINORAMA MOVIES
- The Bad and the Beautiful
- Enhypen (Walk the Line Summer Edition) in Cinemas
- Palestine 36
- Το Αδελφάτο των Ιπποτών της Ελεεινής Τραπέζης
- Άνοιξη, Καλοκαίρι, Φθινόπωρο, Χειμώνας... και Άνοιξη
- Good Luck, Have Fun, Don’t Die
- EPiC: Elvis Presley σε Μια Μοναδική Συναυλία
- Θα Χυθεί Αίμα
- Ο Θάνατος Ενός Γραφειοκράτη
- Θηλυκό
- Μαρσουπιλαμί
- ΜΕΤΡΟΠΟΛΙΤΑΝ
- Το Μίσος
- Μνήμες Υπανάπτυξης
- Μπόμπ Σφουγγαράκης: Η Aναζήτηση του Τετραγωνοπαντελονή
- O Μυστικός Πράκτορας Μπέρνι: Αποστολή στον Άρη
-